In [1]:
!pip uninstall -y torchcodec

In [2]:
!pip install datasets transformers torch soundfile librosa jiwer tqdm

In [3]:
import torch
import numpy as np
import io
import re
import json
import csv
from datetime import datetime
from pathlib import Path
import soundfile as sf
from datasets import load_dataset
from transformers import pipeline
from jiwer import wer, cer
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')


d:\MyProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

#dataset and model configuration
NUM_SAMPLES = 1000  # use 3000 samples for testing
DATASET_SPLIT = "train" 

MODEL_NAME = "openai/whisper-small" 

OUTPUT_JSON = "transcription_results_1000.json"
OUTPUT_CSV = "transcription_results_1000.csv"

# checkpoint configuration
CHECKPOINT_FILE = "checkpoint_1000.json"
SAVE_EVERY = 100  

# if break,whether to continue
RESUME = False  


In [5]:
def normalize_text(text):
    """
    regularize the text for WER/CER calulation
    """
    if not text:
        return ""
    
    # lowercase, remove punctuation
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    # 合并多个空格
    text = re.sub(r'\s+', ' ', text)
    
    # 去除首尾空格
    return text.strip()

## 5. 检查点管理

In [6]:
def save_checkpoint(results, index):
    """保存检查点"""
    checkpoint = {
        'last_index': index,
        'total_processed': len(results),
        'timestamp': datetime.now().isoformat(),
        'results': results
    }
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, ensure_ascii=False)

def load_checkpoint():
    """加载检查点"""
    try:
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        return None

print("✓ 检查点管理函数已定义")

✓ 检查点管理函数已定义


## 6. 加载数据集

In [7]:

full_dataset = load_dataset(
    "speechbrain/LoquaciousSet",
    "small",
    split=DATASET_SPLIT
)
#choose subset for testing
dataset = full_dataset.select(range(min(NUM_SAMPLES, len(full_dataset))))

print(f"✓ 数据集加载完成")
print(f"  总可用样本: {len(full_dataset)}")
print(f"  选取样本: {len(dataset)}")
print(f"  数据集分割: {DATASET_SPLIT}")

✓ 数据集加载完成
  总可用样本: 107303
  选取样本: 1000
  数据集分割: train


## 7. 初始化模型

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用设备: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\n正在加载模型: {MODEL_NAME}...")
transcriber = pipeline(
    "automatic-speech-recognition",
    model=MODEL_NAME,
    device=0 if device == "cuda" else -1,
    chunk_length_s=30
)

使用设备: cpu

正在加载模型: openai/whisper-small...


Loading weights: 100%|██████████| 479/479 [00:00<00:00, 644.24it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


## 8. 音频处理和转录

In [9]:
'''def decode_audio(wav_dict, target_sr=16000):
    """解码音频"""
    try:
        audio_bytes = wav_dict['bytes']
        audio_array, sample_rate = sf.read(
            io.BytesIO(audio_bytes),
            dtype='float32'
        )
        
        if sample_rate != target_sr:
            import librosa
            audio_array = librosa.resample(
                audio_array,
                orig_sr=sample_rate,
                target_sr=target_sr
            )
        
        return {'array': audio_array, 'sampling_rate': target_sr}
    except Exception as e:
        print(f"\n音频解码失败: {e}")
        return None'''

def decode_audio(wav_dict, target_sr=16000):
    """
    使用 soundfile 替代 torchcodec 处理音频字节流
    """
    try:
        # 确保是从 wav_dict['bytes'] 读取
        audio_bytes = wav_dict['bytes']
        
        # 使用 soundfile 直接从内存字节流读取
        # 这一步完全不经过 FFmpeg，非常安全
        with io.BytesIO(audio_bytes) as f:
            audio_array, sample_rate = sf.read(f, dtype='float32')
        
        # 处理多声道（如果有的话）
        if len(audio_array.shape) > 1:
            audio_array = audio_array.mean(axis=1)
            
        # 重采样
        if sample_rate != target_sr:
            import librosa
            audio_array = librosa.resample(
                audio_array,
                orig_sr=sample_rate,
                target_sr=target_sr
            )
        
        return {'array': audio_array, 'sampling_rate': target_sr}
    except Exception as e:
        print(f"\n音频解码失败: {e}")
        return None

'''def transcribe_sample(sample):
    """转录单个样本"""
    try:
        audio_data = decode_audio(sample['wav'])
        if audio_data is None:
            return ""
        
        result = transcriber(audio_data)
        return result["text"].strip()
    except Exception as e:
        return ""'''

def transcribe_sample(sample):
    try:
        audio_data = decode_audio(sample['wav'])
        if audio_data is None:
            print("音频解码返回 None")
            return ""
        result = transcriber(audio_data)
        return result["text"].strip()
    except Exception as e:
        print(f"转录过程中出错: {e}") # 看看这里到底报了什么错
        return ""

## 9. 主转录流程

In [10]:
# 检查断点
start_idx = 0
results = []

if RESUME:
    checkpoint = load_checkpoint()
    if checkpoint:
        results = checkpoint['results']
        start_idx = checkpoint['last_index'] + 1
        print(f"✓ 从检查点恢复: 已处理 {len(results)} 个样本")
        print(f"  从第 {start_idx + 1} 个样本继续\n")

# 开始转录
print("="*80)
print(f"开始转录: {start_idx + 1} 到 {len(dataset)} (共 {len(dataset) - start_idx} 个样本)")
print("="*80)
print()

success_count = sum(1 for r in results if r['success'])

for i in tqdm(range(start_idx, len(dataset)), 
              initial=start_idx,
              total=len(dataset),
              desc="转录进度"):
    
    sample = dataset[i]
    
    # 获取参考文本
    reference_original = sample['text']
    
    # 转录
    prediction_original = transcribe_sample(sample)
    
    # 判断成功
    success = bool(prediction_original)
    if success:
        success_count += 1
    
    # 保存结果
    result = {
        'index': i,
        'id': sample['ID'],
        'reference_text': reference_original,
        'transcription': prediction_original,
        'success': success
    }
    results.append(result)
    
    # 定期保存检查点
    if (i + 1) % SAVE_EVERY == 0:
        save_checkpoint(results, i)
        tqdm.write(f"✓ 检查点已保存 ({i + 1}/{len(dataset)})")

# 完成
print("\n" + "="*80)
print("转录完成!")
print("="*80)
print(f"总样本数: {len(results)}")
print(f"成功转录: {success_count} ({success_count/len(results)*100:.1f}%)")
print(f"失败数量: {len(results) - success_count}")

开始转录: 1 到 1000 (共 1000 个样本)



转录进度:   0%|          | 0/1000 [00:00<?, ?it/s]Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see r

✓ 检查点已保存 (100/1000)


转录进度:  20%|██        | 200/1000 [1:02:15<2:09:21,  9.70s/it]    

✓ 检查点已保存 (200/1000)


...

✓ 检查点已保存 (800/1000)


转录进度:  90%|█████████ | 900/1000 [3:58:23<20:42, 12.43s/it]    

✓ 检查点已保存 (900/1000)


转录进度: 100%|██████████| 1000/1000 [4:26:22<00:00, 15.98s/it]   

✓ 检查点已保存 (1000/1000)

转录完成!
总样本数: 1000
成功转录: 1000 (100.0%)
失败数量: 0


## 10. 计算整体 WER 和 CER

In [11]:

# 收集成功的样本（用于规范化文本计算）
references_norm = []
predictions_norm = []

for result in results:
    if result['success']:
        ref_norm = normalize_text(result['reference_text'])
        pred_norm = normalize_text(result['transcription'])
        
        references_norm.append(ref_norm)
        predictions_norm.append(pred_norm)

# 计算 WER 和 CER
if len(predictions_norm) > 0:
    wer_score = wer(references_norm, predictions_norm)
    cer_score = cer(references_norm, predictions_norm)
    
    print(f"\n基于 {len(predictions_norm)} 个成功样本:")
    print(f"  WER (词错误率): {wer_score:.4f} ({wer_score*100:.2f}%)")
    print(f"  CER (字符错误率): {cer_score:.4f} ({cer_score*100:.2f}%)")
    
    # 保存指标
    metrics = {
        'wer': wer_score,
        'cer': cer_score,
        'total_samples': len(results),
        'successful_samples': len(predictions_norm),
        'success_rate': len(predictions_norm) / len(results)
    }
else:
    print("\n⚠️  没有成功的转录样本")
    metrics = None


基于 1000 个成功样本:
  WER (词错误率): 1.1576 (115.76%)
  CER (字符错误率): 0.9875 (98.75%)


## 11. 保存为 JSON

In [12]:
# 构建输出数据
output_data = {
    'metadata': {
        'created_at': datetime.now().isoformat(),
        'dataset': 'speechbrain/LoquaciousSet',
        'split': DATASET_SPLIT,
        'model': MODEL_NAME,
        'total_samples': len(results),
        'successful_transcriptions': success_count,
        'metrics': metrics
    },
    'results': results
}

# 保存 JSON
print(f"\n正在保存 JSON: {OUTPUT_JSON}...")
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

file_size = Path(OUTPUT_JSON).stat().st_size / (1024 * 1024)
print(f"✓ JSON 已保存")
print(f"  文件: {Path(OUTPUT_JSON).absolute()}")
print(f"  大小: {file_size:.2f} MB")


正在保存 JSON: transcription_results_1000.json...
✓ JSON 已保存
  文件: d:\MyProject\transcription_results_1000.json
  大小: 0.61 MB


## 12. 保存为 CSV

In [13]:
print(f"\n正在保存 CSV: {OUTPUT_CSV}...")

with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['index', 'id', 'reference_text', 'transcription', 'success']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    
    writer.writeheader()
    writer.writerows(results)

file_size = Path(OUTPUT_CSV).stat().st_size / (1024 * 1024)
print(f"✓ CSV 已保存")
print(f"  文件: {Path(OUTPUT_CSV).absolute()}")
print(f"  大小: {file_size:.2f} MB")


正在保存 CSV: transcription_results_1000.csv...
✓ CSV 已保存
  文件: d:\MyProject\transcription_results_1000.csv
  大小: 0.50 MB


## 13. 结果预览

In [14]:
print("\n" + "="*80)
print("结果预览")
print("="*80)

print("\n【前3个成功样本】")
success_samples = [r for r in results if r['success']][:3]

for i, sample in enumerate(success_samples, 1):
    print(f"\n样本 {i}:")
    print(f"  ID: {sample['id']}")
    print(f"  原文: {sample['reference_text'][:80]}...")
    print(f"  转录: {sample['transcription'][:80]}...")
    
    # 计算该样本的匹配度
    ref_norm = normalize_text(sample['reference_text'])
    pred_norm = normalize_text(sample['transcription'])
    
    if ref_norm == pred_norm:
        print(f"  匹配: ✓ 完全匹配")
    else:
        sample_wer = wer(ref_norm, pred_norm)
        print(f"  匹配: WER {sample_wer:.2%}")


结果预览

【前3个成功样本】

样本 1:
  ID: 20091124-0900-PLENARY-18-en_20091124-22:35:55_9
  原文: AND WHAT ABOUT INTEROPERABILITY IN THE RAIL SECTOR ARE NATIONAL BARRIERS PREVENT...
  转录: Rydyn ni'n meddwl i'r cyfnod i'r cyfnod i'r cyfnod i'r cyfnod i'r cyfnod i'r cyf...
  匹配: WER 355.56%

样本 2:
  ID: 20171024-0900-PLENARY-19-en_20171024-19:35:50_20
  原文: THIS WILL NOT HAPPEN THIS IS A PROMISE I MAKE ON BEHALF OF THE COMMISSION AND TH...
  转录: will not happen. This is a promise I make on behalf of the Commission. And this ...
  匹配: WER 2.38%

样本 3:
  ID: 20150520-0900-PLENARY-8-en_20150520-13:27:12_5
  原文: IT IS GOOD TO SEE SOMETHING BEING DONE ABOUT IT BUT THE IDEA THAT THE EUROPEAN U...
  转录: It's good to see something being done about it, but the idea that the European U...
  匹配: WER 22.50%


## 14. 最终总结

In [15]:
print("\n" + "="*80)
print("Phase 1 完成总结")
print("="*80)

print(f"\n✓ 转录任务完成")
print(f"  处理样本: {len(results)}")
print(f"  成功率: {success_count/len(results)*100:.1f}%")

if metrics:
    print(f"\n✓ 评估指标 (规范化后)")
    print(f"  WER: {metrics['wer']*100:.2f}%")
    print(f"  CER: {metrics['cer']*100:.2f}%")

print(f"\n✓ 输出文件")
print(f"  JSON: {OUTPUT_JSON}")
print(f"  CSV: {OUTPUT_CSV}")

print(f"\n下一步: Phase 2 - 词频分析和关键词提取")
print(f"  运行 'Phase2_Analysis.ipynb'")

# 清理检查点
try:
    Path(CHECKPOINT_FILE).unlink()
    print(f"\n✓ 检查点文件已清理")
except:
    pass

print("\n" + "="*80)


Phase 1 完成总结

✓ 转录任务完成
  处理样本: 1000
  成功率: 100.0%

✓ 评估指标 (规范化后)
  WER: 115.76%
  CER: 98.75%

✓ 输出文件
  JSON: transcription_results_1000.json
  CSV: transcription_results_1000.csv

下一步: Phase 2 - 词频分析和关键词提取
  运行 'Phase2_Analysis.ipynb'

✓ 检查点文件已清理

